In [2]:
#r "C:\Users\user\Desktop\Kristina\practice2026\task17\bin\Debug\net10.0\task17.dll"
#r "nuget:ScottPlot,5.0.54"

using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;
using System.Threading;
using ScottPlot;
using task17;

public class TestCommand(int id) : ICommand
{
    private int _counter = 0;
    private IScheduler _scheduler;

    public void SetScheduler(IScheduler scheduler) => _scheduler = scheduler;

    public void Execute()
    {
        if (_counter >= 3) return;
        
        _counter++;
        Console.WriteLine($"Поток {id} вызов {_counter}");
        
        if (_counter < 3 && _scheduler != null)
            _scheduler.Add(this);
    }
}

public class HeavyCommand : ICommand
{
    private int _rem;
    private readonly IScheduler _scheduler;
    public bool IsCompleted => _rem <= 0;
    
    public HeavyCommand(int n, IScheduler scheduler = null)
    {
        _rem = n;
        _scheduler = scheduler;
    }

    public void Execute()
    {
        if (_rem <= 0) return;
        
        _rem--;
        double x = 0;
        for (int i = 0; i < 100000; i++)
            x += Math.Sin(i) * Math.Sin(i);
        
        if (_rem > 0 && _scheduler != null)
            _scheduler.Add(this);
    }
}

Console.WriteLine("5 команд по 3 вызова:\n");

var scheduler = new Scheduler();
var server = new ServerThread(scheduler);

for (int id = 1; id <= 5; id++)
{
    var cmd = new TestCommand(id);
    cmd.SetScheduler(scheduler);
    scheduler.Add(cmd);
}

server.Start();
Thread.Sleep(1000);
server.HardStop();
server.Join();

Console.WriteLine("\nПоток остановлен через HardStop\n");

var counts = new int[] { 2, 4, 6, 8, 10 };
var seqTimes = new double[counts.Length];
var rrTimes = new double[counts.Length];

for (int i = 0; i < counts.Length; i++)
{
    int n = counts[i];

    double seqTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var sw = Stopwatch.StartNew();
        for (int j = 0; j < n; j++)
        {
            var cmd = new HeavyCommand(3);
            while (!cmd.IsCompleted) cmd.Execute();
        }
        sw.Stop();
        seqTotal += sw.Elapsed.TotalMilliseconds;
    }
    seqTimes[i] = seqTotal / 3;

    double rrTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var s = new Scheduler();
        var st = new ServerThread(s);
        var cmds = new HeavyCommand[n];
        for (int j = 0; j < n; j++)
        {
            cmds[j] = new HeavyCommand(3, s);
            s.Add(cmds[j]);
        }

        var sw = Stopwatch.StartNew();
        st.Start();
        while (!cmds.All(c => c.IsCompleted)) Thread.Sleep(1);
        sw.Stop();
        st.HardStop();
        st.Join();
        rrTotal += sw.Elapsed.TotalMilliseconds;
    }
    rrTimes[i] = rrTotal / 3;
}

var plot = new Plot();
plot.XLabel("Количество команд");
plot.YLabel("Время (мс)");

var xs = counts.Select(x => (double)x).ToArray();
var s1 = plot.Add.Scatter(xs, seqTimes);
s1.LegendText = "Без планировщика";
s1.LineWidth = 2;
s1.MarkerSize = 8;

var s2 = plot.Add.Scatter(xs, rrTimes);
s2.LegendText = "Round Robin";
s2.LineWidth = 2;
s2.MarkerSize = 8;

plot.ShowLegend();
plot.SavePng("res.png", 800, 600);
display(HTML($"<img src='res.png' width='700'/>"));

Installed Packages ScottPlot, 5.0.54

5 команд по 3 вызова:

Поток 1 вызов 1
Поток 2 вызов 1
Поток 3 вызов 1
Поток 4 вызов 1
Поток 5 вызов 1
Поток 1 вызов 2
Поток 2 вызов 2
Поток 3 вызов 2
Поток 4 вызов 2
Поток 5 вызов 2
Поток 1 вызов 3
Поток 2 вызов 3
Поток 3 вызов 3
Поток 4 вызов 3
Поток 5 вызов 3

Поток остановлен через HardStop

